# Notebook 04 — Why AppSync, why FIBO-shaped

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI Semantic Layer Workshop on AWS — Workshop 2

---

Why does the UI talk to GraphQL instead of calling agents directly?

This notebook answers that question by walking through the FIBO-shaped schema
and the two resolver paths the running system actually uses: a **read path** that
queries Neptune directly for speed, and an **action path** that invokes an
AgentCore agent. It is honest about the authorization boundary too — what is
enforced today (who may call a field) versus what is roadmap (per-row Lake
Formation scoping).

In [ ]:
# WS2 notebook setup — installs only what this notebook needs into the kernel.
# Skips uv sync (which installs the full agent stack and takes minutes).
import sys, subprocess

# Only install packages not already provided by the SageMaker base image
pkgs = ['rdflib>=7.0.0', 'pyshacl>=0.25.0', 'SPARQLWrapper>=2.0.0']

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     '--disable-pip-version-check'] + pkgs,
    cwd='/tmp'
)
print('WS2 dependencies ready.')


## Key terms for this notebook

| Term | What it is |
|------|------------|
| **FIBO-shaped schema** | A GraphQL schema whose types correspond to FIBO classes (or Workshop 1's atlas: extensions of them). A developer writing a UI component writes a fragment against `Customer`, `Household`, `WealthSignal` — not against arbitrary backend types. |
| **Read path (direct Neptune)** | How the running system serves data reads. The resolver Lambda runs **inside the VPC** and queries the Neptune SLGD cluster **directly** over SigV4-signed HTTP — it does **not** go through an MCP server or an agent. This is the fast path (~0.9 s warm) and it is how `customer`, `wealthSignals`, `advisoryRelationships`, `referrals`, `auditTrail` resolve. |
| **Action path (resolver → agent)** | How the running system serves the three *actions*: `askGraph` (→ nl-to-sparql-agent), `draftRationale` (→ referral-rationale-drafter), `converse` (→ conversational-context-manager). The resolver invokes an AgentCore Runtime via `invoke_agent_runtime`. These are not in the VPC and pay the AgentCore cold-start floor (~10 s first call). Routing (`routeReferral`) is a third action: it starts the referral-orchestrator Step Function. |
| **Federation** | One GraphQL query can resolve fields from more than one backend. The UI doesn't know or care which path serves which field — it writes a query against the schema, and the resolver picks read-path or action-path per field. |
| **Persona claim** | The caller's `cognito:group` (e.g. `atlas-consumer-banker`), carried in the JWT. Today it gates **which fields a persona may call** (AppSync Cognito authorization + the agents' `VALID_PERSONAS` allow-lists). It does **not** yet scope **which rows** come back — see the Lake Formation note below. |

## A schema that crosses team boundaries

Notebook 03 registered agents and MCP servers in the Agent Registry. The registry
answers the question *"what capabilities exist?"* But a UI needs more than
capabilities — it needs *data*. A React component that renders a Customer 360
needs to know what fields exist on a Customer, what types those fields have, and
how to fetch them. The Agent Registry does not answer those questions. GraphQL does.

The ATLAS GraphQL schema is FIBO-shaped: every type in the schema maps to exactly
one ontology class from Workshop 1 or Workshop 2. `Customer` maps to `atlas:Customer`.
`AdvisoryRelationship` maps to `atlas:AdvisoryRelationship`. `WealthSignal` maps to
`atlas:WealthSignal`. A developer who knows the ontology can read the schema without
documentation. A developer who knows the schema can navigate the ontology without
a tutorial.

This is also the place where the ontology becomes a contract that crosses team
boundaries. The frontend team writes fragments against FIBO classes. The backend
team implements resolvers that produce instances of FIBO classes. The two teams
can work independently because the schema is the agreement. If the ontology adds
a new class, the schema gets a new type, and the frontend team can write against
it immediately — without waiting for a backend deploy.

### The two paths behind the schema

Behind the schema, the running system uses **two** resolver paths. The split is
about *reads vs. actions*, and it is a deliberate performance decision:

1. **Read path — direct Neptune SPARQL.** For every data read (`customer`,
   `household`, `wealthSignals`, `advisoryRelationships`, `referrals`,
   `auditTrail`), the resolver Lambda runs **inside the VPC** and queries the
   Neptune SLGD cluster **directly** over SigV4-signed HTTP. There is no MCP
   server and no agent in this path. Warm, it returns in well under a second.
   Earlier drafts of ATLAS routed reads through `atlas-sparql-mcp` (and, on paper,
   through Ontop-over-Iceberg); the running system reads Neptune directly because
   the per-call AgentCore invoke floor (~5 s) was too slow to put behind a UI on
   every field. The MCP path remains as a fallback (resolver outside the VPC) and
   for governed writes.

2. **Action path — resolver → AgentCore agent.** For the things that *do* work
   (not just read), the resolver invokes an AgentCore Runtime:
   - `askGraph` → **nl-to-sparql-agent** (template-bounded natural-language → SPARQL),
   - `draftRationale` → **referral-rationale-drafter** (Bedrock, probabilistic, human-in-the-loop),
   - `converse` → **conversational-context-manager** (wraps the same NL→SPARQL agent),
   - `routeReferral` → starts the **referral-orchestrator** Step Function.

   These pay the AgentCore cold-start cost (~10 s on the first call after idle),
   which is why the demo "warms" them once before showtime.

The UI does not know which path serves which field. It writes a GraphQL query;
the resolver picks read-path or action-path per field. This is the abstraction
that makes the UI forward-compatible: when the backend changes (a field moves
from direct-Neptune to a federated source, a new agent is added), the UI keeps
working because the schema hasn't changed.

### What the persona claim does — and does not — do today

It is tempting to say "the same query returns different data for different
personas." Be precise about what is true in the running system:

- **Enforced today (access control):** the persona claim gates *which fields a
  caller may invoke*. AppSync authorizes every request by Cognito group, and the
  action agents carry an explicit `VALID_PERSONAS` allow-list (for example, only
  `atlas-consumer-banker` may call `draftRationale`). A persona that is not
  allowed cannot call the field at all. The MCP servers, when in the path,
  **validate** the persona claim the same way.
- **Roadmap, not enforced (row scoping):** per-row Lake Formation scoping — where
  the *same* field returns a *different set of rows* depending on persona — is
  **not** wired in the running system. The direct-Neptune read path does not pass
  the persona claim down to a row filter; it returns the same rows regardless of
  caller. The design target is Lake Formation-scoped Iceberg via Ontop, but that
  is future work. **Do not claim row-level scoping is live.**

So: persona decides *who can call what* (real, today). Persona deciding *which
rows you see* is the roadmap. Keeping those two apart is the honest version of
the "four-layer permission model" — the access-control layer is enforced; the
data-scoping layer is designed but not yet enforced.

In [ ]:
import sys
import os
import json

# Workshop 1's shared helpers
sys.path.insert(0, "../../../agentic-semantic-layer/notebooks/shared")

# Load the GraphQL schema for inspection
SCHEMA_PATH = "../../spec/05-appsync-graphql/schema.graphql"

with open(SCHEMA_PATH) as f:
    schema_text = f.read()

print(f"Schema loaded: {len(schema_text)} characters")
print(f"Types defined: {schema_text.count('type ')}")

In [ ]:
# Build cell 1 — Inspect the schema types and their ontology mappings.
#
# Every type in the schema has a docstring that names the ontology class
# it maps to. This is the FIBO-shaped contract.

import re

# Extract type definitions with their docstrings
type_pattern = re.compile(r'"""\n(.+?)\n"""\ntype (\w+)', re.DOTALL)
matches = type_pattern.findall(schema_text)

print("GraphQL types and their ontology mappings:\n")
for docstring, type_name in matches:
    # Extract the ontology class from the docstring
    first_line = docstring.strip().split('\n')[0]
    print(f"  {type_name:30s} → {first_line}")

print(f"\nTotal: {len(matches)} typed entities in the schema")

In [ ]:
# Build cell 2 — Simulate the READ-PATH resolver (direct Neptune SPARQL).
#
# This mirrors what the running customer resolver does: it builds a SPARQL
# query and runs it DIRECTLY against the Neptune SLGD cluster (SigV4-signed
# HTTP, from inside the VPC). No MCP server, no agent — that's the fast path.
# We simulate the Neptune response with a local return so the notebook runs
# without VPC access.

from atlas_sparql import prefixed

def resolve_customer(uri):
    """Simulate the Customer resolver (READ PATH: direct Neptune SPARQL).

    Note there is no persona_claim argument: the running read path does NOT
    pass persona down to a row filter. AppSync has already authorized WHETHER
    this caller may invoke the field; the rows returned are the same regardless
    of persona (per-row Lake Formation scoping is roadmap, not enforced).
    """
    sparql = prefixed(f"""
        SELECT ?customerId ?label WHERE {{
            <{uri}> a atlas:Customer ;
                atlas:customerId ?customerId .
            OPTIONAL {{ <{uri}> rdfs:label ?label }}
        }}
    """)

    # In the running system: resolver runs this SPARQL directly against Neptune
    # over SigV4 (neptune-db:ReadDataViaQuery), in-VPC, ~0.9s warm.
    print("  Read path: resolver → Neptune SLGD directly (SigV4, in-VPC)")
    print(f"  SPARQL (first 100 chars): {sparql[:100]}...")

    # Simulated response — in the running system this comes from Neptune.
    return {
        "uri": uri,
        "customerId": "CUST-9C2A1E",
        "label": "Anjali Patel",
    }

# Execute the resolver
result = resolve_customer("atlas:cust/9c2a1e")
print(f"\n  Result: {json.dumps(result, indent=2)}")

In [ ]:
# Build cell 3 — Demonstrate both real paths: read (direct Neptune) + action (→ agent).
#
# Read path:   resolver → Neptune SLGD directly (SigV4, in-VPC). No agent.
#              e.g. customer, wealthSignals, advisoryRelationships, referrals.
# Action path: resolver → AgentCore Runtime (invoke_agent_runtime).
#              e.g. askGraph → nl-to-sparql-agent, draftRationale → drafter,
#              converse → conversational-context-manager; routeReferral → SFN.

def resolve_wealth_signals(customer_uri):
    """READ PATH: direct Neptune SPARQL for a customer's signals (no agent, no persona filter)."""
    sparql = prefixed(f"""
        SELECT ?signal ?signalType ?signalDate WHERE {{
            ?signal a atlas:WealthSignal ;
                atlas:aboutCustomer <{customer_uri}> ;
                atlas:hasSignalType ?signalType .
            OPTIONAL {{ ?signal atlas:signalDate ?signalDate }}
        }}
    """)
    print("  Read path (direct Neptune): resolver → Neptune SLGD, SigV4, in-VPC")
    # Simulated response — signals are DERIVED ATLAS OUTPUTS (see Notebook 05), not inputs.
    return [
        {"uri": "atlas:signal/deposit-001", "signalType": "LargeDepositPattern"},
        {"uri": "atlas:signal/gap-001", "signalType": "NoAdvisorCoverageSignal"},
    ]

def invoke_ask_graph(question):
    """ACTION PATH: askGraph → nl-to-sparql-agent (template-bounded NL→SPARQL)."""
    print(f"  Action path (→ agent): resolver → invoke_agent_runtime(nl-to-sparql-agent)")
    # The agent NEVER free-generates SPARQL; it matches the question to a validated
    # template (cosine >= 0.75) and runs that. An unmatched question → no_template_match.
    return {"status": "success", "templateId": "signals_for_customer", "rows": 2}

print("Two real paths in action:\n")
print("1. Read — graph data (direct Neptune, no agent):")
signals = resolve_wealth_signals("atlas:cust/9c2a1e")
print(f"  Signals found: {len(signals)}")

print("\n2. Action — natural-language query (resolver → nl-to-sparql-agent):")
nl = invoke_ask_graph("What signals does this customer have?")
print(f"  status={nl['status']} template={nl['templateId']} rows={nl['rows']}")

In [ ]:
# Build cell 4 — What the persona claim ACTUALLY does today: access control.
#
# HONEST version. The persona claim decides WHICH FIELDS a caller may invoke
# (enforced today), NOT which rows come back (roadmap). We model the real
# allow-list that the running system enforces — e.g. only the Consumer Banker
# may draftRationale — and show that an unauthorized persona is REFUSED the
# call, not given a smaller result set.

# These allow-lists mirror what the running system enforces: AppSync Cognito
# authorization on every field, plus each action agent's VALID_PERSONAS.
FIELD_ALLOW_LISTS = {
    "draftRationale": ["atlas-consumer-banker"],          # drafter VALID_PERSONAS
    "routeReferral":  ["atlas-consumer-banker"],          # only the banker routes
    "askGraph":       ["atlas-consumer-banker", "atlas-wealth-advisor", "atlas-bsa-analyst"],
}

def authorize(field, persona_claim):
    """Return whether this persona may INVOKE this field (access control)."""
    allowed = FIELD_ALLOW_LISTS.get(field, [])
    return persona_claim in allowed

print("Access control — who may call draftRationale (enforced today):\n")
for persona in ["atlas-consumer-banker", "atlas-wealth-advisor", "atlas-bsa-analyst"]:
    ok = authorize("draftRationale", persona)
    print(f"  {persona:24s} → {'ALLOWED' if ok else 'REFUSED'}")

print("\nNote what this is NOT: a refused persona is denied the CALL, it is not")
print("handed a smaller set of rows. Per-row Lake Formation scoping (same field,")
print("different rows per persona) is ROADMAP — not enforced in the running system.")

## Verification

Three things must be true for the GraphQL federation layer to be correct:

1. Every type in the schema maps to an ontology class (FIBO-shaped contract holds)
2. The read-path and action-path resolvers produce correctly-shaped responses
3. The persona claim gates *which fields* a caller may invoke (access control —
   enforced today). We verify access control, **not** row scoping, because row
   scoping is roadmap and asserting it would be a lie.

In [ ]:
# Verification cell 1 — Every type maps to an ontology class.
#
# We check that every type definition in the schema has a docstring
# that references either atlas: or atlas-part-2: or FIBO.
# If this fails: a type was added without an ontology mapping.

# Types that are infrastructure (not ontology-mapped)
INFRA_TYPES = {"Query", "Mutation", "Subscription", "Provenance", "Capability"}

all_types = re.findall(r'^type (\w+)', schema_text, re.MULTILINE)
ontology_mapped = {name for _, name in matches}

unmapped = set(all_types) - ontology_mapped - INFRA_TYPES

print(f"Total types in schema: {len(all_types)}")
print(f"Ontology-mapped types: {len(ontology_mapped)}")
print(f"Infrastructure types:  {len(INFRA_TYPES)}")
print(f"Unmapped types:        {unmapped if unmapped else 'None'}")

assert len(unmapped) == 0, f"Types without ontology mapping: {unmapped}"
print("\n✓ Every entity type in the schema maps to an ontology class.")

In [ ]:
# Verification cell 2 — Resolver responses match GraphQL type shapes.
#
# We verify that the simulated resolver responses contain the fields
# declared as required (non-nullable) in the schema.
# If this fails: the resolver is not returning all required fields.

# Customer type requires: uri, customerId (READ PATH — no persona argument)
customer_result = resolve_customer("atlas:cust/9c2a1e")
assert "uri" in customer_result, "Customer resolver must return 'uri'"
assert "customerId" in customer_result, "Customer resolver must return 'customerId'"

# WealthSignal requires: uri, signalType (READ PATH)
signal_results = resolve_wealth_signals("atlas:cust/9c2a1e")
for sig in signal_results:
    assert "uri" in sig, "WealthSignal resolver must return 'uri'"
    assert "signalType" in sig, "WealthSignal resolver must return 'signalType'"

# askGraph (ACTION PATH) must return an honest status — never a fabricated row.
nl_result = invoke_ask_graph("What signals does this customer have?")
assert nl_result["status"] in {"success", "no_template_match", "execution_error"}, \
    "askGraph must return an honest status, never invent one"

print("\n✓ Read-path and action-path responses contain the required fields.")

In [ ]:
# Verification cell 3 — The persona claim gates field ACCESS (enforced today).
#
# This proves the access-control layer: only allow-listed personas may invoke
# a protected field. It deliberately does NOT assert row scoping, because the
# running system does not enforce row scoping (that is roadmap). Asserting a
# different ROW COUNT per persona here would test a fiction.
# If this fails: an allow-list drifted from the running system's VALID_PERSONAS.

# Only the Consumer Banker may draft a rationale or route a referral.
assert authorize("draftRationale", "atlas-consumer-banker") is True, \
    "Consumer Banker must be allowed to draftRationale"
assert authorize("draftRationale", "atlas-wealth-advisor") is False, \
    "Wealth Advisor must NOT be allowed to draftRationale"
assert authorize("routeReferral", "atlas-bsa-analyst") is False, \
    "BSA Analyst must NOT be allowed to routeReferral"

# A read/ask field is open to more personas — but still gated by an allow-list,
# not open to everyone implicitly.
assert authorize("askGraph", "atlas-wealth-advisor") is True, \
    "Wealth Advisor should be allowed to askGraph"

print("✓ Access control confirmed: the persona claim decides WHO MAY CALL each field.")
print("  This is the layer that is enforced today. Per-row data scoping is roadmap —")
print("  the read path returns the same rows regardless of persona (do not claim otherwise).")

## What just changed

You have seen the FIBO-shaped GraphQL schema that both UIs consume, and the two
paths behind it in the running system: a **read path** that queries Neptune
**directly** (SigV4, in-VPC, no agent) for speed, and an **action path** that
invokes an AgentCore agent for `askGraph`, `draftRationale`, and `converse` (plus
`routeReferral`, which starts a Step Function). You also saw the honest shape of
the permission model: the persona claim **gates which fields a caller may invoke**
(access control, enforced today), while **per-row Lake Formation scoping is
roadmap, not enforced** — the read path returns the same rows regardless of persona.

This is Thesis 2 (two UIs, one backbone) made concrete: the Wholesale UI and the
Wealth UI both write fragments against this one schema; they differ only in which
fragments they query and how they render the results.

Where to go deeper:
- **The FIBO grounding** of `Customer` / `Account` / `Holding` (the docstrings you
  inspected) is taught in Workshop 1 — see
  [`02_fibo_alignment.ipynb`](../../../agentic-semantic-layer/notebooks/02_fibo_alignment.ipynb)
  (`fibo-fnd-pty-pty:IndependentParty`, `fibo-fbc-pas-fpas:FinancialAccount`).
- **The action agents** behind `askGraph` / `draftRationale` — why they refuse to
  invent SPARQL and draft only from grounded context — are in
  [`03_agent_registry.ipynb`](./03_agent_registry.ipynb) and
  [`04a_how_agents_work.ipynb`](./04a_how_agents_work.ipynb).
- **The signals** the read path serves are *derived ATLAS outputs*, not inputs —
  see [`05_wealth_signals.ipynb`](./05_wealth_signals.ipynb).

The next notebook puts a UI on top of this schema and shows the access-control
layer visible in the rendered application.